In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate einops
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q git+https://github.com/facebookresearch/sam2.git

import os, torch, numpy as np
from PIL import Image
from tqdm import tqdm

BASE     = '/content/drive/MyDrive/SAM2_LatentDiff'
DATASETS = os.path.join(BASE, 'datasets')
PREPROC  = os.path.join(BASE, 'preprocessed')

print('Ready!')

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 120.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 96.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 67.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 44.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.9/142.9 MB 18.4 MB/s eta 0:00:00
   

In [5]:
# from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
# import glob

# # Load SAM (full precision — half() causes dtype errors)
# sam = sam_model_registry['vit_h'](
#     checkpoint=os.path.join(BASE, 'checkpoints/sam/sam_vit_h_4b8939.pth')
# )
# sam = sam.to('cuda')

# # Faster settings
# mask_gen = SamAutomaticMaskGenerator(
#     sam,
#     points_per_side=16,
#     pred_iou_thresh=0.86,
#     stability_score_thresh=0.92,
#     min_mask_region_area=100,
# )

# # Find all low-light input images
# input_dirs = [
#     os.path.join(DATASETS, 'LOLv2_Real/Train/Input'),
#     os.path.join(DATASETS, 'LOLv2_Real/Test/Input'),
#     os.path.join(DATASETS, 'LOLv2_Synthetic/Train/Input'),
#     os.path.join(DATASETS, 'LOLv2_Synthetic/Test/Input'),
# ]

# image_paths = []
# for d in input_dirs:
#     image_paths += sorted(glob.glob(os.path.join(d, '*.png')))
#     image_paths += sorted(glob.glob(os.path.join(d, '*.jpg')))

# print(f'Found {len(image_paths)} low-light images')

# output_dir = os.path.join(PREPROC, 'sam_priors')
# os.makedirs(output_dir, exist_ok=True)

# already_done = len([f for f in os.listdir(output_dir) if f.endswith('.npy')])
# print(f'Already processed: {already_done}, remaining: {len(image_paths) - already_done}')

# for img_path in tqdm(image_paths):
#     name = os.path.splitext(os.path.basename(img_path))[0]
#     save_path = os.path.join(output_dir, f'{name}.npy')

#     if os.path.exists(save_path):
#         continue

#     img = np.array(Image.open(img_path).convert('RGB'))
#     masks = mask_gen.generate(img)
#     np.save(save_path, masks)

# print(f'Done! SAM priors saved to {output_dir}')

# del sam, mask_gen
# torch.cuda.empty_cache()

In [6]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import glob

# Load SAM2
sam2_model = build_sam2(
    'sam2_hiera_l.yaml',
    os.path.join(BASE, 'checkpoints/sam2/sam2_hiera_large.pt'),
    device='cuda'
)
predictor = SAM2ImagePredictor(sam2_model)

# Find all low-light input images
input_dirs = [
    os.path.join(DATASETS, 'LOLv2_Real/Train/Input'),
    os.path.join(DATASETS, 'LOLv2_Real/Test/Input'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Train/Input'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Test/Input'),
]

image_paths = []
for d in input_dirs:
    image_paths += sorted(glob.glob(os.path.join(d, '*.png')))
    image_paths += sorted(glob.glob(os.path.join(d, '*.jpg')))

# Filter only Input (low-light) images
print(f'Found {len(image_paths)} low-light images')

# Output directory
feat_dir = os.path.join(PREPROC, 'sam2_features')
os.makedirs(feat_dir, exist_ok=True)

already_done = len([f for f in os.listdir(feat_dir) if f.endswith('.pt')])
print(f'Already processed: {already_done}, remaining: {len(image_paths) - already_done}')

for img_path in tqdm(image_paths):
    name = os.path.splitext(os.path.basename(img_path))[0]
    save_path = os.path.join(feat_dir, f'{name}.pt')

    if os.path.exists(save_path):
        continue

    img = np.array(Image.open(img_path).convert('RGB'))
    predictor.set_image(img)
    features = predictor.get_image_embedding()
    torch.save(features.cpu(), save_path)

print(f'SAM2 features extracted for {len(image_paths)} images!')

del sam2_model, predictor
torch.cuda.empty_cache()

Found 1789 low-light images
Already processed: 0, remaining: 1789


100%|██████████| 1789/1789 [15:46<00:00,  1.89it/s]

SAM2 features extracted for 1789 images!


In [7]:
from diffusers import AutoencoderKL
from torchvision import transforms
import glob

# Load VAE
vae = AutoencoderKL.from_pretrained(
    os.path.join(BASE, 'checkpoints/vae'),
    torch_dtype=torch.float16
).to('cuda')
vae.eval()

# Image transform
transform = transforms.Compose([
    transforms.Resize((400, 600)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# Output directory
latent_dir = os.path.join(PREPROC, 'vae_latents')
os.makedirs(latent_dir, exist_ok=True)

# Process ALL images (both Input and GT, Train and Test)
all_dirs = [
    os.path.join(DATASETS, 'LOLv2_Real/Train/Input'),
    os.path.join(DATASETS, 'LOLv2_Real/Train/GT'),
    os.path.join(DATASETS, 'LOLv2_Real/Test/Input'),
    os.path.join(DATASETS, 'LOLv2_Real/Test/GT'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Train/Input'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Train/GT'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Test/Input'),
    os.path.join(DATASETS, 'LOLv2_Synthetic/Test/GT'),
]

all_images = []
for d in all_dirs:
    all_images += sorted(glob.glob(os.path.join(d, '*.png')))
    all_images += sorted(glob.glob(os.path.join(d, '*.jpg')))

print(f'Encoding {len(all_images)} images to latent space...')

already_done = len([f for f in os.listdir(latent_dir) if f.endswith('.pt')])
print(f'Already processed: {already_done}, remaining: {len(all_images) - already_done}')

for img_path in tqdm(all_images):
    name = os.path.splitext(os.path.basename(img_path))[0]
    parent = os.path.basename(os.path.dirname(img_path))
    save_path = os.path.join(latent_dir, f'{parent}_{name}.pt')

    if os.path.exists(save_path):
        continue

    img = Image.open(img_path).convert('RGB')
    x = transform(img).unsqueeze(0).half().to('cuda')

    with torch.no_grad():
        latent = vae.encode(x).latent_dist.sample() * 0.18215

    torch.save(latent.cpu(), save_path)

print('All images encoded to latents!')

del vae
torch.cuda.empty_cache()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Encoding 3578 images to latent space...
Already processed: 0, remaining: 3578


100%|██████████| 3578/3578 [21:22<00:00,  2.79it/s]

All images encoded to latents!
